# AlexNet Implementation (Pretrained)
Same task and dataset as the scratch AlexNet notebook, but initialised from ImageNet-pretrained
weights instead of random init. This isolates the effect of pretraining on the same architecture without the architecture
itself also changing (unlike the AlexNet-scratch vs ResNet18-pretrained comparison, which
confounds architecture and pretraining together).
*Dataset:*
- 500 class subset of iNaturalist-2021 (mini)
- 50 images per class (40:10 train-test split)
- Data samples processed in src/data_processing/sampling_dataset.ipynb


## Setup

In this stage, we import the necessary libraries

In [1]:
import csv
import time
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

## Data pipeline

The input layer of AlexNet requirs images to be resized to a 224x224x3 tensor. The following transforms are performed:
1) training set:
   1) random resized crop (plus a smale horizontal scale adjustment)
   2) random horizontal filp
   3) to tensor
   4) normalise
2) validation/test set:
   1) resize
   2) centre crop
   3) to tensor
   4) normalise
The normalisation transform will involve the mean and standard deviation commonly used in ImageNet normalisation techniques

In [2]:
# Image transformation environment variables
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# dataset paths set up
DATA_ROOT = Path("C:/dataset/sampled_500")
TRAIN_PATH = DATA_ROOT / "train_mini"
VALIDATION_PATH = DATA_ROOT / "validation"
TEST_PATH = DATA_ROOT / "val"

# establish transforms for all sub-datasets
train_set_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_test_set_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# load datasets uses these transforms and dataset paths
def load_datasets():
    train_ds = datasets.ImageFolder(TRAIN_PATH, transform=train_set_transform)
    validation_ds = datasets.ImageFolder(VALIDATION_PATH, transform=val_test_set_transform)
    test_ds = datasets.ImageFolder(TEST_PATH, transform=val_test_set_transform)
    
    return train_ds, validation_ds, test_ds


In [3]:
# load datasets 
train_ds, val_ds, test_ds = load_datasets()

print(f"classes found: {len(train_ds.classes)}")
print(f"train images:  {len(train_ds)}")
print(f"val images:    {len(val_ds)}")
print(f"test images:   {len(test_ds)}")

image, label = train_ds[0]
print(f"sample tensor shape: {image.shape}")
print(f"sample label index:  {label} -> class '{train_ds.classes[label]}'")

classes found: 500
train images:  20000
val images:    5000
test images:   5000
sample tensor shape: torch.Size([3, 224, 224])
sample label index:  0 -> class '00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii'


## Model Architecture

As mentioned previously, the input layer is a 224x224x3 tensor. Before convolution layers 2, 3 and 5, Max Pooling also takes place. After all the convolution layers, flattening takes place before entering the final fully connected layers.
Below are the remaining layers of the proposed AlexNet model:
|Layer|Implementation|Kernel Size|Stride|Padding|Output Dimension|
|:---|:---|:---|:---|:---|:----|
|Convolution 1|`Conv2d`|11x11|4|0|54x54x96|
|Max Pooling|`MaxPool2d`|3x3|2|0|26x26x96|
|Convolution 2|`Conv2d`|5x5|1|2|26x26x256|
|Max Pooling|`MaxPool2d`|3x3|2|0|12x12x256|
|Convolution 3|`Conv2d`|3x3|1|1|12x12x384|
|Convolution 4|`Conv2d`|3x3|1|1|12x12x384|
|Convolution 5|`Conv2d`|3x3|1|1|12x12x256|
|Max Pooling|`MaxPool2d`|3x3|2|0|5x5x256|
|Flatten|||||6400|
|Fully Connected Layer 1|`Linear`||||4096|
|Fully Connected Layer 2|`Linear`||||4096|
|Fully Connected Layer 3|`Linear`||||500|

A large risk attached to this model is the large size of the Fully Connected Layers. With millions of parameters at play, the model can easily overfit on a small dataset. To prevent this, dropout is used before the first two fully connected layers. This allows only a select number of nodes in each layer to be used based on a given probability.
This helps significantly reduce the number of parameters used.
For the actual implementation in torch, the convolution layers are described as features, and the fully connected layers are described as classifier.


In [4]:
def build_pretrained_alexnet(num_classes, dropout=0.6):
    weights = models.AlexNet_Weights.IMAGENET1K_V1
    model = models.alexnet(weights=weights)
    
    # set dropout layers in classifier to given dropout val
    model.classifier[0] = nn.Dropout(p=dropout)
    model.classifier[3] = nn.Dropout(p=dropout)

    in_features = model.classifier[6].in_features
    # change final layer from output 1000 to 5000 (num_classes)
    model.classifier[6] = nn.Linear(in_features, num_classes)

    return model

## Training setup

Design decisions, and why they differ from a typical transfer-learning setup:

- **SGD with momentum + Nesterov**, not Adam. Scratch CNNs on small datasets tend to generalise
  better with SGD; Adam often converges faster but overfits harder in this regime.
- **Label smoothing (0.1)** on the loss — softens the training targets slightly, a cheap
  regulariser that helps given how overparameterised this model is relative to 20,000 images.
- **Cosine annealing** learning rate schedule — decays smoothly to zero rather than in steps.
- A **higher starting learning rate** (0.1) than a pretrained model would use (typically ~1e-4),
  since random-init weights need larger updates early on to move away from their starting point.

In [5]:
# setting the torch seed and device for training
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
EPOCHS = 15
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_WORKERS = 0
SEED = 42

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [6]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

model = build_pretrained_alexnet(num_classes=len(train_ds.classes), dropout=0.5).to(device=device)
print(model.classifier)
# freeze feature parameters
for param in model.features.parameters():
    param.requires_grad = False
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimiser = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=1e-3, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=9216, out_features=4096, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=4096, out_features=4096, bias=True)
  (5): ReLU(inplace=True)
  (6): Linear(in_features=4096, out_features=500, bias=True)
)


## Training Loop
Training loop implementation handles which set is being used. Training set enables gradient updates, test set provides no weight updates. 

In [7]:
def run_epoch(model, loader, criterion, optimizer, device, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total

## Running Training and Logging Results

Logs each result to a csv logging file which can be used officially in the report. 

In [8]:
out_dir = Path("runs/scratch")
out_dir.mkdir(parents=True, exist_ok=True)

log_path = out_dir / "log.csv"
with open(log_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr"])

best_val_acc = 0.0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimiser, device, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimiser, device, train=False)
    scheduler.step()

    current_lr = optimiser.param_groups[0]["lr"]
    print(f"epoch {epoch}/{EPOCHS} "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} lr={current_lr:.2e}")

    with open(log_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, train_loss, train_acc, val_loss, val_acc, current_lr])

    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "classes": train_ds.classes,
        "mode": "scratch",
        "val_acc": val_acc,
    }
    torch.save(checkpoint, out_dir / "full_last.pt")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(checkpoint, out_dir / "full_best.pt")

total_time = time.time() - start_time
print(f"\nDone. Best val acc: {best_val_acc:.4f}. Total training time: {total_time/60:.1f} min")

c:\Users\Sergio Insuasti\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


epoch 1/15 train_loss=5.1880 train_acc=0.1200 val_loss=4.0830 val_acc=0.2700 lr=9.89e-04
epoch 2/15 train_loss=3.8039 train_acc=0.3165 val_loss=3.6715 val_acc=0.3456 lr=9.57e-04
epoch 3/15 train_loss=3.3730 train_acc=0.4047 val_loss=3.5094 val_acc=0.3862 lr=9.05e-04
epoch 4/15 train_loss=3.1147 train_acc=0.4708 val_loss=3.4304 val_acc=0.4058 lr=8.35e-04
epoch 5/15 train_loss=2.9231 train_acc=0.5168 val_loss=3.3918 val_acc=0.4108 lr=7.50e-04
epoch 6/15 train_loss=2.7682 train_acc=0.5591 val_loss=3.3525 val_acc=0.4192 lr=6.55e-04
epoch 7/15 train_loss=2.6315 train_acc=0.5970 val_loss=3.3402 val_acc=0.4252 lr=5.52e-04
epoch 8/15 train_loss=2.5343 train_acc=0.6238 val_loss=3.3206 val_acc=0.4286 lr=4.48e-04
epoch 9/15 train_loss=2.4533 train_acc=0.6509 val_loss=3.3169 val_acc=0.4300 lr=3.45e-04
epoch 10/15 train_loss=2.3777 train_acc=0.6760 val_loss=3.3005 val_acc=0.4360 lr=2.50e-04
epoch 11/15 train_loss=2.3271 train_acc=0.6894 val_loss=3.2925 val_acc=0.4428 lr=1.65e-04
epoch 12/15 train_l

## Hyperparameter search ~ Determine best hyperparameters (Optuna)

Manual tuning so far has shown dropout strength matters more than `fc_width` for controlling
the overfitting gap, and both interact with `weight_decay`. Rather than continuing to hand-pick
values, this runs an automated search over the three at once, using this notebook's existing
setup.

Each trial trains a fresh model for a reduced number of epochs and reports back val accuracy.
Optuna's pruner can stop clearly-bad trials early rather than running them to completion, saving
time on a CPU-bound search.

**This does not replace the training loop above** -- it searches for the best config here, then
that config gets plugged into the real training run (either the cell above with updated
hyperparameters, or the full-scale run) afterwards.

In [9]:
# import optuna
# SEARCH_EPOCHS = 5
# N_TRIALS = 10

# def objective(trial):
#     dropout = trial.suggest_float("dropout", 0.4, 0.7)
#     weight_decay = trial.suggest_float("weight_decay", 1e-4, 5e-3, log=True)
#     lr = trial.suggest_float("lr", 5e-4, 5e-3, log=True)

#     set_seed(SEED)
#     trial_model = build_pretrained_alexnet(num_classes=len(train_ds.classes), dropout=dropout).to(device)
#     # no freezing -- full fine-tune, matching the better-performing run

#     trial_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
#     trial_optimiser = torch.optim.SGD(trial_model.parameters(), lr=lr, momentum=0.9,
#                                        weight_decay=weight_decay, nesterov=True)
#     trial_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(trial_optimiser, T_max=SEARCH_EPOCHS)

#     val_acc = 0.0
#     for epoch in range(SEARCH_EPOCHS):
#         run_epoch(trial_model, train_loader, trial_criterion, trial_optimiser, device, train=True)
#         _, val_acc = run_epoch(trial_model, val_loader, trial_criterion, trial_optimiser, device, train=False)
#         trial_scheduler.step()
#         trial.report(val_acc, epoch)
#         if trial.should_prune():
#             raise optuna.TrialPruned()
#     return val_acc

# study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
# study.optimize(objective, n_trials=N_TRIALS)

# print(f"\nBest val acc: {study.best_value:.4f}")
# print(f"Best params:  {study.best_params}")

# trials_df = study.trials_dataframe().sort_values("value", ascending=False)
# trials_df[["number", "value", "params_dropout", "params_momentum", "params_weight_decay", "params_fc_width", "state"]]